In [1]:
# Optional: install required libraries for this lecture
%pip install -q openai


### Step 1: Provide a tiny context
Paste a short paragraph. The bot will only answer from this context and say "I don't know" if missing.


In [2]:
CONTEXT = """
Module 3 teaches GenAI APIs with OpenAI, Gemini, and Anthropic.
Each tutorial is short and focuses on building blocks, using real world example and knowledge from proejcts.
"""


### Step 2: Load API key and create client
We'll use the Chat Completions API. Keep temperature low for factual tone.


In [3]:
import os
from getpass import getpass
from openai import OpenAI

# Enter your OpenRouter API key when prompted.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# You can change this to another OpenRouter-supported model.
MODEL = "openai/gpt-4o-mini"

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
)

print("OpenRouter key loaded:", bool(OPENROUTER_API_KEY))
print("Model:", MODEL)


Enter your OpenRouter API key: ··········
OpenRouter key loaded: True
Model: openai/gpt-4o-mini


### Step 3: Ask a question from the context
If the answer is not in the context, the bot should say: "I don't know."


In [4]:
def answer_question(question: str) -> str:
    system_prompt = (
        "You answer using ONLY the provided context. "
        "If the answer isn't in the context, say 'I don't know.' Keep answers under 40 words."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{CONTEXT}\n\nQuestion: {question}"},
        ],
        temperature=0,
    )
    return response.choices[0].message.content

print(answer_question("What does Module 3 teach?"))


Module 3 teaches GenAI APIs with OpenAI, Gemini, and Anthropic, focusing on building blocks through short tutorials using real-world examples and project knowledge.


### Step 4: Show "I don't know" behavior
Ask something not present in the context and confirm the fallback response.


In [5]:
print(answer_question("What is the capital of the Section 3?"))

I don't know.


### Step 5: Add memory with chat history + for-loop chat
We’ll keep a running `chat_history` so the model remembers prior turns. The loop simulates multiple user turns.


In [6]:
system_prompt = (
    "You answer using ONLY the provided context. "
    "If the answer isn't in the context, say 'I don't know.' Keep answers under 40 words."
)

chat_history = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Context:\n{CONTEXT}"},
]

user_turns = [
    "What does this module cover?",
    "Which providers are mentioned?",
    "What is the capital of France?",
]

for turn in user_turns:
    chat_history.append({"role": "user", "content": turn})
    resp = client.chat.completions.create(
        model=MODEL,
        messages=chat_history,
        temperature=0,
    )
    answer = resp.choices[0].message.content
    print("Q:", turn)
    print("A:", answer)
    print()
    chat_history.append({"role": "assistant", "content": answer})


Q: What does this module cover?
A: Module 3 covers GenAI APIs with OpenAI, Gemini, and Anthropic, focusing on building blocks through short tutorials using real-world examples and project knowledge.

Q: Which providers are mentioned?
A: The providers mentioned are OpenAI, Gemini, and Anthropic.

Q: What is the capital of France?
A: I don't know.



### Step 6: Adding user input to the loop


In [ ]:
chat_history = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Context:\n{CONTEXT}"},
]

while True:
    turn = input("You: ")
    chat_history.append({"role": "user", "content": turn})
    resp = client.chat.completions.create(
        model=MODEL,
        messages=chat_history,
        temperature=0,
    )
    answer = resp.choices[0].message.content
    print("Q:", turn)
    print("A:", answer)
    print()
    chat_history.append({"role": "assistant", "content": answer})

You: coder
Q: coder
A: I don't know.

You: historian
Q: historian
A: I don't know.

